### 1. conda 환경 생성 및 활성화
터미널에서 실행:  
conda create -n autogluon_env python=3.10 -y

conda activate autogluon_env

이후 환경 안에서 AutoGluon 설치:

pip install -U pip wheel setuptools

pip install autogluon.tabular -q 


In [1]:
import torch, lightgbm, xgboost, catboost, fastai
print("🔥 All Model Libraries Installed OK")

🔥 All Model Libraries Installed OK


In [2]:
# 파일: autogluon_run.py (예시)

import pandas as pd
from autogluon.tabular import TabularPredictor

# 1) 데이터 로드
train_path = "../data/train.tsv"
test_path = "../data/test.tsv"

# TSV 이므로 sep="\t" 사용
train_df = pd.read_csv(train_path, sep="\t")
test_df = pd.read_csv(test_path, sep="\t")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Train shape: (1482535, 8)
Test shape: (693359, 7)


In [3]:

# 2) 타깃 / ID 컬럼 이름 설정
TARGET_COL = "price"   # 실제 타깃 컬럼명으로 바꿔 주세요
ID_COL = None           # 제출용 ID 컬럼명으로 바꿔 주세요 (없으면 None)

# 3) AutoGluon 학습
# presets / time_limit 등은 상황에 맞게 조절
predictor = TabularPredictor(
    label=TARGET_COL,
    problem_type=None,          # 회귀/분류 자동 추론, 명시하고 싶으면 "regression"/"multiclass"/"binary"
    path="autogluon_models"     # 모델이 저장될 폴더
).fit(
    train_data=train_df,
    presets="medium_quality_faster_train",  # 빠른 실험용
    time_limit=3600,                        # 최대 1시간 (초 단위), 필요에 따라 조정
)


Preset alias specified: 'medium_quality_faster_train' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.10.19
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.19045
CPU Count:          12
Memory Avail:       22.20 GB / 31.91 GB (69.6%)
Disk Space Avail:   325.99 GB / 465.09 GB (70.1%)
Presets specified: ['medium_quality_faster_train']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 3600s
AutoGluon will save models to "c:\big20\git\big20-ML-project2-team3\MercariPriceSuggestion\src\autogluon_models"
Train Data Rows:    1482535
Train Data Columns: 7
Label Column:       price
AutoGluon infers your prediction problem is: 'regression' (because dtype of label-column == float and label-values can't be converted to int).
	Label info (max, min, mean, stddev): (2009.0, 0.0, 26.73752, 38.58607)
	If 'regressi

[1000]	valid_set's rmse: 27.5545
[2000]	valid_set's rmse: 27.2789
[3000]	valid_set's rmse: 27.1386
[4000]	valid_set's rmse: 27.0199
[5000]	valid_set's rmse: 26.9748
[6000]	valid_set's rmse: 26.9484
[7000]	valid_set's rmse: 26.9381
[8000]	valid_set's rmse: 26.9131
[9000]	valid_set's rmse: 26.8819
[10000]	valid_set's rmse: 26.8632


	-26.8591	 = Validation score   (-root_mean_squared_error)
	316.28s	 = Training   runtime
	3.55s	 = Validation runtime
Fitting model: LightGBM ... Training model for up to 2697.89s of the 2697.89s of remaining time.
	Fitting with cpus=6, gpus=0, mem=7.8/18.6 GB


[1000]	valid_set's rmse: 27.1385
[2000]	valid_set's rmse: 26.9219
[3000]	valid_set's rmse: 26.8469
[4000]	valid_set's rmse: 26.8217
[5000]	valid_set's rmse: 26.7739
[6000]	valid_set's rmse: 26.7265
[7000]	valid_set's rmse: 26.6878
[8000]	valid_set's rmse: 26.6515
[9000]	valid_set's rmse: 26.648
[10000]	valid_set's rmse: 26.6327


	-26.6295	 = Validation score   (-root_mean_squared_error)
	317.89s	 = Training   runtime
	3.07s	 = Validation runtime
Fitting model: RandomForestMSE ... Training model for up to 2375.88s of the 2375.88s of remaining time.
	Fitting with cpus=12, gpus=0, mem=0.9/18.5 GB


KeyboardInterrupt: 

In [3]:

# 4) 리더보드 확인 (optional)
leaderboard_df = predictor.leaderboard(silent=True)
print(leaderboard_df.head())


                 model  score_val              eval_metric  pred_time_val  \
0      RandomForestMSE -29.774283  root_mean_squared_error       0.093989   
1  WeightedEnsemble_L2 -29.774283  root_mean_squared_error       0.095003   

      fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  \
0  2325.451227                0.093989        2325.451227            1   
1  2325.456228                0.001014           0.005001            2   

   can_infer  fit_order  
0       True          1  
1       True          2  


In [6]:

# 5) 테스트 데이터 예측
test_preds = predictor.predict(test_df)


KeyError: "1 required columns are missing from the provided dataset to transform using AutoMLPipelineFeatureGenerator. 1 missing columns: ['train_id'] | 7 available columns: ['test_id', 'name', 'item_condition_id', 'category_name', 'brand_name', 'shipping', 'item_description']"

In [ ]:

# 6) 제출 파일 생성
if ID_COL in test_df.columns:
    submission = pd.DataFrame({
        ID_COL: test_df[ID_COL],
        TARGET_COL: test_preds
    })
else:
    # ID 컬럼이 없다면 단순히 index 기반으로 생성
    submission = pd.DataFrame({
        "id": range(len(test_preds)),
        TARGET_COL: test_preds
    })

submission_path = "submission_autogluon.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)
